In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import itertools

from sentence_transformers import SentenceTransformer, util # For Cosine similarity
from bert_score import score # For BERT score
from rouge_score import rouge_scorer # For ROUGE
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction # For BLEU

/home/dferna3/.conda/envs/env_ft_llama2/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:
input_csv_path = 'results.csv'  # Replace with your input CSV file path
output_csv_path = 'comparison_results.csv'  # Desired output CSV file path
#df = pd.read_csv(input_csv_path, encoding = "ISO-8859-1")# does not work
#df = pd.read_csv(input_csv_path, engine='python')
df = pd.read_csv(input_csv_path)
df.iloc[:,1:] = df.iloc[:,1:].astype(str)

In [3]:
# Comparisson functions
cos_sim_model = SentenceTransformer("all-miniLM-L6-v2")
def cosine_similarity(text1, text2):
    text1_embedding = cos_sim_model.encode(text1, convert_to_tensor=True)
    text2_embedding = cos_sim_model.encode(text2, convert_to_tensor=True)
    return round(util.pytorch_cos_sim(text1_embedding, text2_embedding).item(), 2)

def bert_score(text1, text2):
    P, R, F1 = score([text1], [text2], lang="en", verbose=False)
    return round(F1[0].item(),2)

def rouge1_score(text1, text2):
    # Initialize the ROUGE scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(text1, text2)
    return round(scores["rouge1"].fmeasure, 2)

def rouge2_score(text1, text2):
    # Initialize the ROUGE scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(text1, text2)
    return round(scores["rouge2"].fmeasure, 2)

def rougeL_score(text1, text2):
    # Initialize the ROUGE scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(text1, text2)
    return round(scores["rougeL"].fmeasure, 2)

def bleu_score(text1, text2):
    candidate = text2.split() # Tokenize
    bleu_score = sentence_bleu(text1, candidate, smoothing_function=SmoothingFunction().method1)
    return round(bleu_score,4)

comparison_functions = [
    ("cosine_similarity", cosine_similarity),
    ("bert_score", bert_score),
    ("ROUGE_1", rouge1_score),
    ("ROUGE_2", rouge2_score),
    ("ROUGE_L", rougeL_score),
    ("BLEU", bleu_score)
]

/home/dferna3/.conda/envs/env_ft_llama2/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Models vs ground truth

In [4]:
results = []

for _,row in df.iterrows():
    scenario_id = row["scenario_id"]
    ground_truth = row["ground_truth"]
    
    for model in df.columns[2:]:
        model_output = row[model]
        
        # Compute similarity scores
        scores = {
            "scenario_id": scenario_id,
            "model": model,
            "cosine_similarity": cosine_similarity(ground_truth, model_output),
            "bert_score": bert_score(ground_truth, model_output),
            "ROUGE_1": rouge1_score(ground_truth, model_output),
            "ROUGE_2": rouge2_score(ground_truth, model_output),
            "ROUGE_L": rougeL_score(ground_truth, model_output),
            "BLEU": bleu_score(ground_truth, model_output),
        }
        
        results.append(scores)

results_df = pd.DataFrame(results)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

In [5]:
results_df

,scenario_id,model,cosine_similarity,bert_score,ROUGE_1,ROUGE_2,ROUGE_L,BLEU
0,1,gemini-1.5-flash,0.74,0.88,0.48,0.22,0.34,0.0036
1,1,gemini-1.5-flash-8b,0.85,0.87,0.39,0.14,0.24,0.0031
2,1,gemini-2.0-flash-exp,0.75,0.89,0.45,0.16,0.23,0.0031
3,1,gemini-1.5-pro,0.83,0.88,0.40,0.14,0.24,0.0040
4,1,meta-llama/Llama-3.2-1B-Instruct,0.82,0.86,0.41,0.14,0.24,0.0015
...,...,...,...,...,...,...,...,...
364,41,meta-llama/Llama-3.2-1B-Instruct,0.47,0.81,0.07,0.02,0.07,0.0004
365,41,meta-llama/Llama-3.2-3B-Instruct,0.40,0.80,0.08,0.02,0.05,0.0006
366,41,Qwen/Qwen2.5-7B-Instruct-1M,0.25,0.81,0.14,0.06,0.09,0.0007
367,41,Qwen/Qwen2.5-14B-Instruct-1M,0.25,0.81,0.13,0.04,0.08,0.0006


In [6]:
# Model-Wise averages
# Group the resutls by model and calculate mean for each metric

average_results_df = results_df.drop(columns=["scenario_id"]).groupby("model").mean().reset_index()
average_results_df

,model,cosine_similarity,bert_score,ROUGE_1,ROUGE_2,ROUGE_L,BLEU
0,Qwen/Qwen2.5-14B-Instruct-1M,0.280732,0.813171,0.144634,0.044634,0.091707,0.000627
1,Qwen/Qwen2.5-7B-Instruct-1M,0.280732,0.812927,0.150488,0.047805,0.094146,0.000680
2,gemini-1.5-flash,0.680244,0.882439,0.341951,0.117805,0.248537,0.003263
3,gemini-1.5-flash-8b,0.670732,0.875122,0.308780,0.100000,0.215854,0.002766
4,gemini-1.5-pro,0.691463,0.881951,0.330732,0.119024,0.245366,0.003010
5,gemini-2.0-flash-exp,0.677073,0.882439,0.319756,0.111220,0.235122,0.003027
6,meta-llama/Llama-3.2-1B-Instruct,0.618049,0.842195,0.215610,0.061951,0.148049,0.001529
7,meta-llama/Llama-3.2-3B-Instruct,0.530000,0.830488,0.179512,0.053415,0.124390,0.002051
8,meta-llama/Llama-3.2-8B-Instruct,0.477073,0.812927,0.142439,0.045366,0.099512,0.000756


In [7]:
averages_output_file = "modelsVsGt_averages.csv"
average_results_df.to_csv(averages_output_file, index=False)

In [8]:
ai_output_columns = df.columns[1:] 
ai_output_columns

Index(['ground_truth', 'gemini-1.5-flash', 'gemini-1.5-flash-8b',
       'gemini-2.0-flash-exp', 'gemini-1.5-pro',
       'meta-llama/Llama-3.2-1B-Instruct', 'meta-llama/Llama-3.2-3B-Instruct',
       'Qwen/Qwen2.5-7B-Instruct-1M', 'Qwen/Qwen2.5-14B-Instruct-1M',
       'meta-llama/Llama-3.2-8B-Instruct'],
      dtype='object')

## Correlation matrices

In [9]:
# Save to csv file:
#output_df.to_csv(output_csv_path, index=False)